In [ ]:
import yfinance as yf
import pandas as pd

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)


In [2]:
tickers = ["AAPL", "BB", "IRS", "QQQ"]
prices = {}

for ticker_symbol in tickers:
    ticker = yf.Ticker(ticker_symbol)
    prices[ticker_symbol] = ticker.history(start="2009-02-14", end="2020-06-11")
    print(f"\n{ticker_symbol}:")
    print(prices[ticker_symbol])


AAPL:
                                Open       High        Low      Close  \
Date                                                                    
2009-02-17 00:00:00-05:00   2.902516   2.907609   2.824912   2.832402   
2009-02-18 00:00:00-05:00   2.847984   2.871954   2.778171   2.827609   
2009-02-19 00:00:00-05:00   2.797647   2.824014   2.699967   2.715848   
2009-02-20 00:00:00-05:00   2.678693   2.768582   2.666707   2.732626   
2009-02-23 00:00:00-05:00   2.746109   2.756596   2.592100   2.605283   
...                              ...        ...        ...        ...   
2020-06-04 00:00:00-04:00  78.593211  78.891211  77.718577  78.091690   
2020-06-05 00:00:00-04:00  78.341246  80.376396  78.312173  80.315826   
2020-06-08 00:00:00-04:00  80.012971  80.824610  79.303092  80.790688   
2020-06-09 00:00:00-04:00  80.470869  83.734373  80.439372  83.341881   
2020-06-10 00:00:00-04:00  84.289205  85.953667  83.850679  85.486069   

                              Volume  Divid

In [3]:
#Collect Features, we want them a day behind, we give yesterdays stats to guess today's price, so score follows current day
for ticker_symbol in tickers:
    price = prices[ticker_symbol]

    price["Change"]      = price["Close"].shift(1) - price["Open"].shift(1)
    price["Score"]       = (price["Close"] > price["Open"]).astype(int)
    price["%Change"]     = (price["Close"].shift(1) - price["Open"].shift(1)) / price["Open"].shift(1)
    price["CloseToOpen"] = price["Open"].shift(1) - price["Close"].shift(1)
    price["YestScore"]   = price["Score"].shift(1)
    price["5DayMean"]    = price["Close"].rolling(5).mean().shift(1)
    price["5DayVoli"]    = price["Close"].rolling(5).std().shift(1)
    price["5DayPerf"]    = price["Score"].rolling(5).sum().shift(1)

    price.dropna(inplace=True)
    prices[ticker_symbol] = price
    print(f"\n{ticker_symbol}:")
    print(price["5DayPerf"])


AAPL:
Date
2009-02-24 00:00:00-05:00    1.0
2009-02-25 00:00:00-05:00    2.0
2009-02-26 00:00:00-05:00    3.0
2009-02-27 00:00:00-05:00    3.0
2009-03-02 00:00:00-05:00    3.0
                            ... 
2020-06-04 00:00:00-04:00    4.0
2020-06-05 00:00:00-04:00    3.0
2020-06-08 00:00:00-04:00    4.0
2020-06-09 00:00:00-04:00    4.0
2020-06-10 00:00:00-04:00    4.0
Name: 5DayPerf, Length: 2844, dtype: float64

BB:
Date
2009-02-24 00:00:00-05:00    0.0
2009-02-25 00:00:00-05:00    1.0
2009-02-26 00:00:00-05:00    2.0
2009-02-27 00:00:00-05:00    2.0
2009-03-02 00:00:00-05:00    3.0
                            ... 
2020-06-04 00:00:00-04:00    3.0
2020-06-05 00:00:00-04:00    4.0
2020-06-08 00:00:00-04:00    4.0
2020-06-09 00:00:00-04:00    4.0
2020-06-10 00:00:00-04:00    3.0
Name: 5DayPerf, Length: 2844, dtype: float64

IRS:
Date
2009-02-24 00:00:00-05:00    2.0
2009-02-25 00:00:00-05:00    3.0
2009-02-26 00:00:00-05:00    3.0
2009-02-27 00:00:00-05:00    3.0
2009-03-02 00:00:00

In [4]:

features = ["Change", "%Change", "CloseToOpen", "YestScore", "5DayMean", "5DayVoli", "5DayPerf"]
all_results = {}

for ticker_symbol in tickers:
    price = prices[ticker_symbol]
    priceTrain = price[price.index <= "2018-06-11"]
    priceTest = price[price.index > "2018-06-11"]

    results = []

    for i in range(len(priceTest)):
        xTrain = priceTrain[features]
        yTrain = priceTrain["Score"]

        scaler = StandardScaler()
        xTrain = scaler.fit_transform(xTrain)

        model = MLPClassifier(hidden_layer_sizes=(32,), max_iter=1000, random_state=42)
        model.fit(xTrain, yTrain)

        # Grab yesterdays information to predict today's, data already transformed so i is yesterday
        yest = priceTest.iloc[[i]]
        yestScal = scaler.transform(yest[features])
        pred = model.predict(yestScal)[0]
        prob = model.predict_proba(yestScal)[0][1]
        print("Analyzing Day: ", priceTest.index[i])

        results.append({"Date": priceTest.index[i],
                        "Score": priceTest["Score"].iloc[i],
                        "Prediction": pred,
                        "Probability": prob,
                        "Open": priceTest["Open"].iloc[i],
                        "Close": priceTest["Close"].iloc[i]})

        priceTrain = pd.concat([priceTrain, yest])

    all_results[ticker_symbol] = pd.DataFrame(results)

Analyzing Day:  2018-06-12 00:00:00-04:00
Analyzing Day:  2018-06-13 00:00:00-04:00
Analyzing Day:  2018-06-14 00:00:00-04:00
Analyzing Day:  2018-06-15 00:00:00-04:00
Analyzing Day:  2018-06-18 00:00:00-04:00
Analyzing Day:  2018-06-19 00:00:00-04:00
Analyzing Day:  2018-06-20 00:00:00-04:00
Analyzing Day:  2018-06-21 00:00:00-04:00
Analyzing Day:  2018-06-22 00:00:00-04:00
Analyzing Day:  2018-06-25 00:00:00-04:00
Analyzing Day:  2018-06-26 00:00:00-04:00
Analyzing Day:  2018-06-27 00:00:00-04:00
Analyzing Day:  2018-06-28 00:00:00-04:00
Analyzing Day:  2018-06-29 00:00:00-04:00
Analyzing Day:  2018-07-02 00:00:00-04:00
Analyzing Day:  2018-07-03 00:00:00-04:00
Analyzing Day:  2018-07-05 00:00:00-04:00
Analyzing Day:  2018-07-06 00:00:00-04:00
Analyzing Day:  2018-07-09 00:00:00-04:00
Analyzing Day:  2018-07-10 00:00:00-04:00
Analyzing Day:  2018-07-11 00:00:00-04:00
Analyzing Day:  2018-07-12 00:00:00-04:00
Analyzing Day:  2018-07-13 00:00:00-04:00
Analyzing Day:  2018-07-16 00:00:0

In [12]:
for ticker_symbol, results in all_results.items():
    score = results["Score"]
    pred = results["Prediction"]
    op = results["Open"]
    close = results["Close"]

    accuracy  = accuracy_score(score, pred)
    precision = precision_score(score, pred, zero_division=0)
    recall    = recall_score(score, pred, zero_division=0)
    f1        = f1_score(score, pred, zero_division=0)

    print(f"\n{ticker_symbol}:")
    print(" Mean:",results["Score"].mean(), "Accuracy: ", accuracy, "\n  Precision: ", precision, "\n  Recall: ", recall, "\n  F1: ", f1)



AAPL:
 Mean: 0.5506958250497018 Accuracy:  0.5387673956262425 
  Precision:  0.574750830564784 
  Recall:  0.6245487364620939 
  F1:  0.5986159169550173

BB:
 Mean: 0.46322067594433397 Accuracy:  0.5228628230616302 
  Precision:  0.4666666666666667 
  Recall:  0.21030042918454936 
  F1:  0.28994082840236685

IRS:
 Mean: 0.48111332007952284 Accuracy:  0.48906560636182905 
  Precision:  0.46445497630331756 
  Recall:  0.4049586776859504 
  F1:  0.4326710816777042

QQQ:
 Mean: 0.5308151093439364 Accuracy:  0.48906560636182905 
  Precision:  0.5135869565217391 
  Recall:  0.7078651685393258 
  F1:  0.5952755905511811


# Comments
AAPL and QQQ perform best with accuracy around 54% and F1 near 0.60, showing the model is picking up some signal but nothing strong. BB has very low recall (0.21) meaning it's heavily underpredicting up days. Generally the model isn't overfitting badly given train/test are close in performance, but it's also not extracting much signal, suggesting that the features might not express enough. Prediction: There may be improvemnet with sentimental data.

In [11]:
#Find out how much you would have made/lost with this method
for ticker_symbol, results in all_results.items():
    score = results["Score"]
    pred = results["Prediction"]
    op = results["Open"]
    close = results["Close"]

    totalIn = totalOut = totalInvested = totalStoodOut = alltradein = alltradeout = perftradein = perftradeout = 0

    for i, o, c, s in zip(pred.values, op.values, close.values, score.values):
        if i == 1:
            totalIn += 100
            totalInvested += 1
            temp = ((c - o)/o) * 100
            totalOut += 100 + temp
        else:
            totalStoodOut += 1
        alltradein += 100
        alltradeout += (((c-o)/o) * 100) + 100
        if s == 1:
            perftradein += 100
            perftradeout += (((c-o)/o) * 100) + 100

    print(f"\n{ticker_symbol}:")
    print(f"  Model:    Invested ${totalIn:.0f}, Returned ${totalOut:.2f}, Profit ${totalOut - totalIn:.2f} ({totalInvested} trades, {totalStoodOut} skipped)")
    print(f"  Buy&Hold: Invested ${alltradein:.0f}, Returned ${alltradeout:.2f}, Profit ${alltradeout - alltradein:.2f}")
    print(f"  Perfect:  Invested ${perftradein:.0f}, Returned ${perftradeout:.2f}, Profit ${perftradeout - perftradein:.2f}")


AAPL:
  Model:    Invested $30100, Returned $30176.13, Profit $76.13 (301 trades, 202 skipped)
  Buy&Hold: Invested $50300, Returned $50379.01, Profit $79.01
  Perfect:  Invested $27700, Returned $28006.00, Profit $306.00

BB:
  Model:    Invested $10500, Returned $10499.40, Profit $-0.60 (105 trades, 398 skipped)
  Buy&Hold: Invested $50300, Returned $50223.65, Profit $-76.35
  Perfect:  Invested $23300, Returned $23692.35, Profit $392.35

IRS:
  Model:    Invested $21100, Returned $21095.82, Profit $-4.18 (211 trades, 292 skipped)
  Buy&Hold: Invested $50300, Returned $50250.79, Profit $-49.21
  Perfect:  Invested $24200, Returned $24868.27, Profit $668.27

QQQ:
  Model:    Invested $36800, Returned $36828.32, Profit $28.32 (368 trades, 135 skipped)
  Buy&Hold: Invested $50300, Returned $50325.43, Profit $25.43
  Perfect:  Invested $26700, Returned $26914.48, Profit $214.48


# Comments

It appears that MLP beats or matches buy & hold on all 4 while making fewer trades which is pretty good. BB and IRS are rough but the model loses way less than just holding. Perfect ceiling is pretty low at $300-600 on $50k which may mean that the data just doesn't have much signal to extract regardless of the model.